In [ ]:
import itertools

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model_comparison.visualize import plot_jitter_analysis, plot_link_stability
from src.model_comparison.aggregate_results import aggregate_experiment_results

# 1. Hyperparameter Tuning

### Fixed Parameters

| Parameter        | Value  |
|:-----------------|:-------|
| `batch`          | 32     |
| `imgsz`          | 640    |
| `workers`        | 2      |
| `warmup_epochs`  | 3.0    |
| `warmup_bias_lr` | 0.0001 |
| `epochs`         | 50     |
| `patience`       | 20     |
| `classes`        | `[0]`  |

---

### Search Space (Hyperparameter Tuning)

| Parameter | Distribution Type | Range / Values    |
| :--- | :--- |:------------------|
| `freeze` | `choice` | `[11, 23]`        |
| `lr0` | `uniform` | `0.00001` to `0.1` |
| `lrf` | `uniform` | `0.01` to `1.0`   |
| `momentum` | `uniform` | `0.6` to `0.98`   |
| `pose` | `uniform` | `10` to `25`      |
| `weight_decay` | `uniform` | `0.0` to `0.001`   |


In total 60 experiments were ran using the Ray Tune package using the search space defined as shown above. Out of them 18 ran for 50 epochs, the rest were stopped early.

In [ ]:
hyperparameter_tuning_base_dir = "./../runs/train/yolo26l_tune2/"
hyperparameter_tuning_summary_path = f"{hyperparameter_tuning_base_dir}/summary.csv"

hyperparameter_tuning_summary = pd.read_csv(hyperparameter_tuning_summary_path)
top_sets = hyperparameter_tuning_summary.head(4)

In [ ]:
all_trial_data = []

for trial_id in top_sets['trial_id']:
    file_path = f"{hyperparameter_tuning_base_dir}/_tune_{trial_id}/progress.csv"
    df_trial = pd.read_csv(file_path)
    df_trial['trial_id'] = f"Trial_{trial_id}"
    all_trial_data.append(df_trial)

In [ ]:
tuning_df = pd.concat(all_trial_data, ignore_index=True)

fig, axes = plt.subplots(3, 2, figsize=(30, 18))
fig.suptitle("Hyperparameter Tuning Performance: Top Trials Comparison", fontsize=16)


sns.lineplot(data=tuning_df, x='epoch', y='metrics/mAP50-95(P)', hue='trial_id', ax=axes[0, 0], linewidth=2)
axes[0, 0].set_title("mAP50-95(P)")
axes[0, 0].set_xlabel("epoch")
axes[0, 0].set_ylabel("metrics/mAP50-95(P)")
axes[0, 0].grid(True, linestyle='--', alpha=0.6)

sns.lineplot(data=tuning_df, x='epoch', y='metrics/mAP50(P)', hue='trial_id', ax=axes[0,1 ], linewidth=2)
axes[0, 1].set_title("mAP50(P)")
axes[0, 1].set_xlabel("epoch")
axes[0, 1].set_ylabel("metrics/mAP50(P)")
axes[0, 1].grid(True, linestyle='--', alpha=0.6)

sns.lineplot(data=tuning_df, x='epoch', y='val/pose_loss', hue='trial_id', ax=axes[1, 1], linewidth=2)
axes[1, 1].set_title("validation pose loss")
axes[1, 1].set_xlabel("epoch")
axes[1, 1].set_ylabel("validation pose loss")
axes[1, 1].grid(True, linestyle='--', alpha=0.6)

sns.lineplot(data=tuning_df, x='epoch', y='val/box_loss', hue='trial_id', ax=axes[1, 0], linewidth=2)
axes[1, 0].set_title("validation box loss")
axes[1, 0].set_xlabel("epoch")
axes[1, 0].set_ylabel("validation box loss")
axes[1, 0].grid(True, linestyle='--', alpha=0.6)

sns.lineplot(data=tuning_df, x='epoch', y='metrics/mAP50-95(B)', hue='trial_id', ax=axes[2, 1], linewidth=2)
axes[2, 1].set_title("metrics/mAP50-95(B)")
axes[2, 1].set_xlabel("epoch")
axes[2, 1].set_ylabel("metrics/mAP50-95(B)")
axes[2, 1].grid(True, linestyle='--', alpha=0.6)

sns.lineplot(data=tuning_df, x='epoch', y='metrics/mAP50(B)', hue='trial_id', ax=axes[2, 0], linewidth=2)
axes[2, 0].set_title("metrics/mAP50(B)")
axes[2, 0].set_xlabel("epoch")
axes[2, 0].set_ylabel("metrics/mAP50(B)")
axes[2, 0].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(rect=(0, 0.03, 1, 0.95))
plt.show()

# 2. Further Training of the Top 4 Models

Top 4 models were chosen for further fine-tuning. Their training parameters are shown below. Each model was trained for 300 epochs with patience set to 100.

In [ ]:
parameters = ['freeze', 'weight_decay', 'lr0', 'lrf', 'momentum', 'pose']

top_sets[["trial_id", *[f'config/{parameter}' for parameter in parameters]]]

In [ ]:
results = {
    '00033': pd.read_csv('../data/models/yolo/tuned/yolo26l_full_d725f_000332/results.csv'),
    '00034': pd.read_csv('../data/models/yolo/tuned/yolo26l_full_d725f_000342/results.csv'),
    '00041': pd.read_csv('../data/models/yolo/tuned/yolo26l_tune_d725f_000412/results.csv'),
    '00044': pd.read_csv('../data/models/yolo/tuned/yolo26l_tune_d725f_000442/results.csv'),
}

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = [
    'train/pose_loss',
    'val/pose_loss',
    'metrics/mAP50(P)',
    'metrics/mAP50-95(P)',
    'metrics/mAP50(B)',
    'metrics/mAP50-95(B)'
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes_flat = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    ax = axes_flat[i]

    for model_id, df in results.items():
        df.columns = df.columns.str.strip()

        x_axis = df['epoch'] if 'epoch' in df.columns else df.index
        ax.plot(x_axis, df[metric], label=f"Model {model_id}", linewidth=2)

    ax.set_title(metric.replace('/', ' / '), fontsize=14, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Score / Loss', fontsize=12)
    ax.legend(loc='best')
    ax.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# 3. Model Stability Comparison

This analysis is restricted to the sequence of frames bounded by the first and last fully detected frames.

* **Jitter**
  Jitter quantifies unnatural, high-acceleration movements for each detected keypoint. It is evaluated using two primary metrics:
  * **Outlier Percentage:** Identifies extreme frame-to-frame jumps. Outliers are statistically defined using the Interquartile Range (IQR) method as any value exceeding $Q_3 + 1.5 \times \text{IQR}$.
  * **Magnitude:** Represents the severity of the jitter, defined as the value at the 95th percentile ($q_{0.95}$) of acceleration.

* **Link Coefficient of Variation (CV)**
  This metric evaluates the consistency of the skeletal structure over time. For every frame, the pixel distance (length) of each relevant link between two connected keypoints is measured. The stability of this link length across the sequence is then calculated using the Coefficient of Variation:

  $$CV = \frac{\sigma}{\mu}$$

  *(where $\sigma$ is the standard deviation of the link length and $\mu$ is the mean link length).*

* **Number of Gaps**
  This tracks missing spatial data. It is defined as the total count of frames within the analyzed window where the model failed to detect either the entire person or specific individual keypoints.

In [ ]:
def get_paths_for_model(model_id):
    environments = ['outdoor', 'treadmill']
    configs = [('RES1080_FPS60_LENlinear', 60), ('RES1080_FPS240_LENwide', 240)]
    versions = ['v1.json', 'v2.json']

    paths = {}
    for env, (folder, fps), ver in itertools.product(environments, configs, versions):
        path = f"../data/output/tuned_yolo/{model_id}/{env}/{folder}/{ver}"
        paths[path] = fps

    return paths

In [ ]:
YOLO_LINKS = {
    # Torso
    "Torso_L": ("left_shoulder", "left_hip"),
    "Torso_R": ("right_shoulder", "right_hip"),

    # Legs
    "Thigh_L": ("left_hip", "left_knee"),
    "Thigh_R": ("right_hip", "right_knee"),
    "Shin_L": ("left_knee", "left_ankle"),
    "Shin_R": ("right_knee", "right_ankle"),

    # Feet
    "Foot_L": ("left_heel", "left_big_toe"),
    "Foot_R": ("right_heel", "right_big_toe"),
}

In [ ]:
model_ids = ['000332', '000342', '000412', '000442']
metrics_for_model = {}

for model_id in model_ids:
    paths = get_paths_for_model(model_id)
    links, jitter, gaps = aggregate_experiment_results(paths, model_id, YOLO_LINKS)
    metrics_for_model[model_id] = {
        "links": links,
        "jitter": jitter,
        'gaps': gaps,
    }

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes_flat = axes.flatten()

for i, model_id in enumerate(model_ids):
    jitter = metrics_for_model[model_id]['jitter']
    plot_jitter_analysis(jitter, axes_flat[i], model_name=f"Model {model_id}")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

axes_flat = axes.flatten()

for i, model_id in enumerate(model_ids):
    links = metrics_for_model[model_id]['links']
    plot_link_stability(links, axes_flat[i], model_name=f"Model {model_id}")

plt.tight_layout()
plt.show()

In [ ]:
mean_gaps_data = {}

for model_id, metrics in metrics_for_model.items():
    mean_gaps_data[model_id] = metrics['gaps']['gap_count'].mean()
df_mean_gaps = pd.DataFrame.from_dict(mean_gaps_data, orient='index', columns=['mean_gaps'])
df_mean_gaps.index.name = 'model_id'

df_mean_gaps